# Person identifier (OpenCV + face_recognition)

Run cells in order. Install deps if needed: `pip install opencv-python face_recognition`

In [ ]:
# 1) Init: face_recognition + in-memory person database
import cv2
import numpy as np
import face_recognition

# Euclidean distance on 128-d encodings; lower = more alike (typical match < 0.6)
MATCH_THRESHOLD = 0.55


class PersonRegistry:
    """Stores one face encoding per assigned person id."""

    def __init__(self, match_threshold=MATCH_THRESHOLD):
        self.match_threshold = match_threshold
        self._next_id = 1
        self.persons = []  # list of {"id": int, "encoding": np.ndarray}

    def identify(self, encoding):
        enc = np.asarray(encoding, dtype=np.float64).flatten()
        if len(self.persons) == 0:
            pid = self._next_id
            self._next_id += 1
            self.persons.append({"id": pid, "encoding": enc.copy()})
            return pid
        known = [p["encoding"] for p in self.persons]
        distances = face_recognition.face_distance(known, enc)
        j = int(np.argmin(distances))
        if distances[j] <= self.match_threshold:
            return self.persons[j]["id"]
        pid = self._next_id
        self._next_id += 1
        self.persons.append({"id": pid, "encoding": enc.copy()})
        return pid


db = PersonRegistry()
print("PersonRegistry ready. Run the next cell for live capture (q to quit).")

In [ ]:
# 2) Live VideoCapture: bbox + person id (press q to quit)
WINDOW = "Person ID (q to quit)"
SCALE = 0.5  # resize for faster detection; boxes are scaled back to full frame
PROCESS_EVERY_N = 2  # run face detection every N frames

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Could not open camera index 0")

cv2.namedWindow(WINDOW, cv2.WINDOW_NORMAL)
frame_i = 0
last_detections = []  # (x1, y1, x2, y2, pid) in full-frame coordinates

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame_i += 1
        h, w = frame.shape[:2]

        if frame_i % PROCESS_EVERY_N == 0:
            small = cv2.resize(frame, (0, 0), fx=SCALE, fy=SCALE, interpolation=cv2.INTER_AREA)
            rgb = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)
            locs = face_recognition.face_locations(rgb, model="hog")
            encs = face_recognition.face_encodings(rgb, locs) if locs else []
            last_detections = []
            for loc, enc in zip(locs, encs):
                top, right, bottom, left = loc
                x1 = int(left / SCALE)
                y1 = int(top / SCALE)
                x2 = int(right / SCALE)
                y2 = int(bottom / SCALE)
                pid = db.identify(enc)
                last_detections.append((x1, y1, x2, y2, pid))

        vis = frame.copy()
        for (x1, y1, x2, y2, pid) in last_detections:
            cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 200, 0), 2)
            cv2.putText(
                vis,
                f"ID {pid}",
                (x1, max(0, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 200, 0),
                2,
            )
        cv2.imshow(WINDOW, vis)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()

print("Stopped.")

In [ ]:
# 3) Print database contents
import json

rows = []
for p in db.persons:
    e = np.asarray(p["encoding"], dtype=np.float64)
    rows.append(
        {
            "id": p["id"],
            "encoding_dim": int(e.size),
            "encoding_preview": np.round(e[:8], 4).tolist(),
        }
    )
print(json.dumps(rows, indent=2))
print(f"\nTotal persons stored: {len(db.persons)}")